# Day 3 · Notebook 6 — Capstone: End-to-End Pipeline

**Objectives**
- Chain everything from the week into one function: acquire → process → extract mask → save
- Run it as a batch over a small date range, with one metadata row per `image_id`
- Practice the **upsert pattern**: re-running for a date already processed updates that row instead of duplicating it
- See where this scales beyond the training ROI — DEM cross-validation, time-series joins, predictive modeling

This notebook doesn't introduce new GEE concepts — it's about assembling Monday–Wednesday's pieces into something that behaves like a real pipeline, not a sequence of one-off cells.

## Setup — shared config

In [ ]:
import os
import json
import ee
import geemap
import rasterio
import rasterio.features
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime
from scipy import ndimage
from skimage import filters, morphology
from shapely.geometry import shape

PROJECT_ID = "YOUR_PROJECT_ID"
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

ROI = ee.Geometry.Polygon([
    [36.021644575238554, 0.15353594768753728],
    [36.18025968754324, 0.15353594768753728],
    [36.18025968754324, 0.36364721537653627],
    [36.021644575238554, 0.36364721537653627],
    [36.021644575238554, 0.15353594768753728],
])

REGION_NAME = "bogoria"
UTM_CRS = "EPSG:32637"
OUTPUT_DIR = Path(f"./training/{REGION_NAME}")
(OUTPUT_DIR / "raw").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "processed").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "metadata").mkdir(parents=True, exist_ok=True)

## 1. One function per pipeline stage

Same three stages as the week: **acquire**, **process**, **extract**. Each takes an `image_id` (derived from `end_date`, same convention used all week) and returns what the next stage needs — this is deliberately close to how `ImageAcquisition` / `SARProcessor` are structured in the real project, just flattened into functions for training clarity.

In [ ]:
def acquire_sentinel1(roi, start_date, end_date, region_name):
    image_id = int(end_date.replace("-", ""))
    s1_collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.eq("orbitProperties_pass", "ASCENDING"))
        .filter(ee.Filter.eq("instrumentMode", "IW"))
    )

    count = s1_collection.size().getInfo()
    if count == 0:
        print(f"  ⚠️ No Sentinel-1 scenes for {start_date} → {end_date}")
        return None, image_id

    def preprocess_sar(image):
        edge = image.lt(-30.0)
        masked = image.mask().And(edge.Not())
        return image.updateMask(masked).clip(roi)

    composite = s1_collection.map(preprocess_sar).median()

    out_path = OUTPUT_DIR / "raw" / f"{region_name}_{image_id}_VV_ASCENDING.tif"
    geemap.ee_export_image(composite, filename=str(out_path), scale=10, region=roi, crs=UTM_CRS)

    if not (out_path.exists() and out_path.stat().st_size > 0):
        print(f"  ✗ Export failed or empty for {image_id}")
        return None, image_id

    print(f"  ✅ Acquired {image_id} ({count} scenes composited)")
    return str(out_path), image_id

In [ ]:
def process_sar(image_path, region_name, image_id, threshold_method="otsu"):
    with rasterio.open(image_path) as src:
        image = src.read(1)
        profile = src.profile
        transform = src.transform
        crs = src.crs

    assert not crs.is_geographic, f"{image_id}: mask CRS must be projected, got {crs}"

    mean = ndimage.uniform_filter(image, size=7)
    mean_sq = ndimage.uniform_filter(image**2, size=7)
    variance = mean_sq - mean**2
    overall_variance = ndimage.variance(image)
    weights = variance / (variance + overall_variance)
    filtered = mean + weights * (image - mean)

    if threshold_method == "otsu":
        threshold = filters.threshold_otsu(filtered)
    else:
        raise ValueError(f"Unsupported threshold_method: {threshold_method}")

    water_mask = filtered < threshold
    water_mask = morphology.remove_small_objects(water_mask, min_size=100)
    water_mask = morphology.closing(water_mask, morphology.disk(3))
    water_mask = morphology.opening(water_mask, morphology.disk(2))

    out_path = OUTPUT_DIR / "processed" / f"{region_name}_{image_id}_water_mask.tif"
    out_profile = profile.copy()
    out_profile.update({"dtype": "uint8", "count": 1, "compress": "lzw", "nodata": 0})
    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(water_mask.astype("uint8"), 1)

    total_pixels = image.size
    water_pixels = int(water_mask.sum())
    print(f"  ✅ Processed {image_id}: {water_pixels}/{total_pixels} water pixels")

    return {
        "mask_path": str(out_path),
        "transform": transform,
        "crs": crs,
        "total_pixels": total_pixels,
        "water_pixels": water_pixels,
    }

In [ ]:
def extract_boundary(mask_path, transform, crs, image_id, min_area_m2=20000, simplify_tolerance=10):
    with rasterio.open(mask_path) as src:
        water_mask = src.read(1)

    shapes_gen = rasterio.features.shapes(water_mask.astype("uint8"), transform=transform)
    geometries = [shape(geom) for geom, value in shapes_gen if value == 1]

    if not geometries:
        print(f"  ⚠️ No water polygons for {image_id}")
        return None

    gdf = gpd.GeoDataFrame({"geometry": geometries}, crs=crs)
    gdf["area_m2"] = gdf.geometry.area
    gdf = gdf[gdf["area_m2"] >= min_area_m2].reset_index(drop=True)

    if gdf.empty:
        print(f"  ⚠️ All polygons filtered out for {image_id}")
        return None

    gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance)
    gdf["area_m2"] = gdf.geometry.area
    gdf["perimeter_m"] = gdf.geometry.length
    gdf["compactness"] = ((4 * np.pi * gdf["area_m2"]) / (gdf["perimeter_m"] ** 2)).clip(0, 1)

    print(f"  ✅ Extracted {len(gdf)} boundary polygon(s), total area {gdf['area_m2'].sum():,.0f} m²")
    return gdf

## 2. Metadata — write JSON + CSV, upsert on `image_id`

Same pattern used across every acquisition/processing class in the project: a running JSON log (append-only, full history) plus a CSV summary where **each `image_id` gets exactly one row** — re-running the pipeline for a date you've already processed replaces that row rather than duplicating it.

In [ ]:
def save_metadata(region_name, image_id, results: dict):
    metadata_dir = OUTPUT_DIR / "metadata"

    json_path = metadata_dir / f"{region_name}_pipeline.json"
    json_entry = {**results, "image_id": image_id, "run_time": datetime.now().isoformat()}

    if json_path.exists():
        with open(json_path, "r") as f:
            existing = json.load(f)
            existing = existing if isinstance(existing, list) else [existing]
    else:
        existing = []
    existing.append(json_entry)
    with open(json_path, "w") as f:
        json.dump(existing, f, indent=2, default=str)

    csv_path = metadata_dir / f"{region_name}_pipeline.csv"
    df_entry = pd.DataFrame([{
        "image_id": image_id,
        "water_pixels": results.get("water_pixels", ""),
        "total_pixels": results.get("total_pixels", ""),
        "boundary_area_m2": results.get("boundary_area_m2", ""),
        "boundary_perimeter_m": results.get("boundary_perimeter_m", ""),
        "mask_path": results.get("mask_path", ""),
    }])

    if csv_path.exists():
        existing_df = pd.read_csv(csv_path)
        existing_df = existing_df[existing_df["image_id"] != image_id]  # drop old row for this id
        updated_df = pd.concat([existing_df, df_entry], ignore_index=True)
        updated_df.to_csv(csv_path, index=False)
    else:
        df_entry.to_csv(csv_path, index=False)

    print(f"  📝 Metadata saved ({csv_path.name})")

## 3. The full pipeline, run over a small batch

Three monthly dates, same ROI. This mirrors the batch pattern used across the project's `batch_*` methods — loop over dates, call each stage, save metadata, continue on failure rather than stopping the whole run.

In [ ]:
end_dates = ["2023-11-30", "2023-12-31", "2024-01-31"]

for end_date in end_dates:
    start_date = end_date[:8] + "01"  # first of the same month
    print(f"\n📅 Processing {start_date} → {end_date}")

    try:
        image_path, image_id = acquire_sentinel1(ROI, start_date, end_date, REGION_NAME)
        if image_path is None:
            continue

        sar_results = process_sar(image_path, REGION_NAME, image_id)
        boundary_gdf = extract_boundary(
            sar_results["mask_path"], sar_results["transform"], sar_results["crs"], image_id
        )

        save_metadata(REGION_NAME, image_id, {
            "total_pixels": sar_results["total_pixels"],
            "water_pixels": sar_results["water_pixels"],
            "mask_path": sar_results["mask_path"],
            "boundary_area_m2": boundary_gdf["area_m2"].sum() if boundary_gdf is not None else "",
            "boundary_perimeter_m": boundary_gdf["perimeter_m"].sum() if boundary_gdf is not None else "",
        })

    except Exception as e:
        print(f"  ❌ Failed for {end_date}: {e}")
        continue

print("\n✅ Batch complete")

In [ ]:
# Inspect the accumulated metadata CSV — one row per image_id
metadata_csv = OUTPUT_DIR / "metadata" / f"{REGION_NAME}_pipeline.csv"
pd.read_csv(metadata_csv)

**Try it**: re-run the cell above for just `end_dates = ["2024-01-31"]`. Check the CSV again — you should still see exactly one row for that `image_id`, with a fresh `run_time` in the JSON log, not a duplicate row in the CSV. That's the upsert pattern working.

## 4. Where this goes from here

This week covered acquisition → processing → mask extraction for one region, one sensor pair (S1 + S2), over three months. The production Riftwaters pipeline extends the same pattern in a few directions, worth knowing about even though we won't build them today:

- **DEM cross-validation**: pair each SAR-derived water extent with elevation from the Copernicus GLO-30 DEM to estimate volume, not just area — and cross-check SAR/optical boundaries against DEM-implied shorelines.
- **Climate drivers**: join ERA5 and CHIRPS precipitation/evaporation time series to each `image_id`'s date, to relate water extent changes to rainfall.
- **Land cover exposure**: use Dynamic World to quantify what land cover type gets inundated as the lake expands (useful for the flood-exposure/roads work).
- **Unified feature table**: one row per date/region joining all of the above — SAR metrics, WBM stats, spectral index stats, land cover exposure, DEM-derived volume, climate variables — which becomes the direct input to a predictive model.
- **Multi-region, multi-lake**: everything here generalizes to Naivasha, Nakuru, and beyond — the only region-specific detail to watch for each time is the **UTM zone**.

If you want to keep working with this after training: the `dataset/{region}/raw|processed|metadata/` folder structure and `{region}_{image_id}` naming convention you used all week is exactly what the full project uses — nothing here was training-only scaffolding.

## Exercise 6.1 (open-ended)

Extend the pipeline above to also acquire and index Sentinel-2 for each date (Notebook 2/3 logic), and add MNDWI-derived `boundary_area_m2` as an extra column in the metadata CSV alongside the SAR-derived one — so each row lets you compare both sources at a glance, the way you did manually in Notebook 5.

In [ ]:
# Your solution here
